In [1]:
import os
os.environ["PYOPENGL_PLATFORM"] = "egl"
import pyrender
import cv2
import glob
import json
import time
import trimesh
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(suppress=True, precision=3)
from bpc.inference.utils.camera_utils import load_camera_params
from bpc.inference.process_pose import PoseEstimator, PoseEstimatorParams, load_pose_model
import bpc.utils.data_utils as du
from bpc.utils.data_utils import Capture, render_mask
from bpc.inference.yolo_detection import YOLODetector, ObjectDetector
import bpc.inference.yolo_detection_filtering as ydf


In [2]:
PHASE = 2 # 1 for phase 1, 2 for phase 2 of the OpenCV BPC challenge.

# Set the paths to the scene and models directories:

if PHASE == 1:
    scene_dir = "/home/joao/source/bpc-challenge/opencv/bpc/ipd/phase-1-backup/val/000001"
    models_dir = '/home/joao/source/bpc-challenge/opencv/bpc/ipd/phase-1-backup/models_eval'
elif PHASE == 2:
    scene_dir = "//mnt/061A31701A315E3D/ipd-dataset/bpc_baseline/datasets/phase2/train_pbr/000001"
    models_dir = '/mnt/061A31701A315E3D/ipd-dataset/bpc_baseline/datasets/phase2/models_eval'

cam_ids = ["cam1", "cam2", "cam3"]

# Get image IDs from the scene directory
image_ids = []
for filename in os.listdir(os.path.join(scene_dir, "rgb_cam1")):
    if filename.endswith(".png") or filename.endswith(".jpg"):
        image_id = int(filename.split(".")[0])
        image_ids.append(image_id)
image_ids = sorted(image_ids)  # Ensure the list is sorted

obj_ids_phase1 = [0, 1, 4, 8, 10, 11, 14, 18, 19, 20]
obj_ids_phase2 = [id for id in range(0, 10)]

if PHASE == 1:
    obj_ids = obj_ids_phase1
elif PHASE == 2:
    obj_ids = obj_ids_phase2


DEPTH_IMAGE_SCALE_PX2MM = 0.1 # from depth pixel raw value to millimeters


In [ ]:
val_depth_image_dir = "/mnt/061A31701A315E3D/ipd-dataset/bpc_baseline/datasets/phase1/val/000000/depth_cam1"
train_pbr_depth_image_dir = "/mnt/061A31701A315E3D/ipd-dataset/bpc_baseline/datasets/phase1/train_pbr/000049/depth_cam1"
val_depth_image_filename = "000003.png"
train_pbr_depth_image_filename = "000006.png"
val_depth_image = cv2.imread(os.path.join(val_depth_image_dir, val_depth_image_filename), flags=cv2.IMREAD_UNCHANGED)
train_pbr_depth_image = cv2.imread(os.path.join(train_pbr_depth_image_dir, train_pbr_depth_image_filename), flags=cv2.IMREAD_UNCHANGED)

val_hillshade_img_0 = du.hillshade_depth_image(val_depth_image, azimuth=0, is_synthetic=False)
val_hillshade_img_135 = du.hillshade_depth_image(val_depth_image, azimuth=135, is_synthetic=False)

train_pbr_hillshade_img_0 = du.hillshade_depth_image(train_pbr_depth_image, azimuth=0, is_synthetic=True)
train_pbr_hillshade_img_135 = du.hillshade_depth_image(train_pbr_depth_image, azimuth=135, is_synthetic=True)

plt.figure(figsize=(20, 10))
plt.subplot(1, 2, 1)
plt.imshow(val_hillshade_img_0, cmap='gray')
plt.title("Hillshade image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(val_hillshade_img_135, cmap='gray')
plt.title("Hillshade image")
plt.axis("off")
plt.show()

plt.figure(figsize=(20, 10))
plt.subplot(1, 2, 1)
plt.imshow(train_pbr_hillshade_img_0, cmap='gray')
plt.title("Hillshade image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(train_pbr_hillshade_img_135, cmap='gray')
plt.title("Hillshade image")
plt.axis("off")
plt.show()


In [ ]:
train_pbr_depth_image_dir = "/mnt/061A31701A315E3D/ipd-dataset/bpc_baseline/datasets/phase1/train_pbr/000049/depth_cam1"


for img_id in range(0, 5):
    train_pbr_depth_image_filename = f"{img_id:06d}.png"
    train_pbr_depth_image = cv2.imread(os.path.join(train_pbr_depth_image_dir, train_pbr_depth_image_filename), flags=cv2.IMREAD_UNCHANGED)
    train_pbr_hillshade_img_0 = du.hillshade_depth_image(train_pbr_depth_image, azimuth=0, is_synthetic=True)
    train_pbr_hillshade_img_135 = du.hillshade_depth_image(train_pbr_depth_image, azimuth=135, is_synthetic=True)
    plt.figure(figsize=(20, 10))
    plt.subplot(1, 2, 1)
    plt.imshow(train_pbr_hillshade_img_0, cmap='gray')
    plt.title(f"Hillshade image - {img_id} - azimuth 0")
    plt.axis("off")
    plt.subplot(1, 2, 2)
    plt.imshow(train_pbr_hillshade_img_135, cmap='gray')
    plt.title(f"Hillshade image - {img_id} - azimuth 135")
    plt.axis("off")
    plt.show()


In [ ]:
depth_image_dir = "/mnt/061A31701A315E3D/ipd-dataset/bpc_baseline/datasets/phase1/val/000006/depth_cam1"
depth_image_filename = "000000.png"
depth_image = cv2.imread(os.path.join(depth_image_dir, depth_image_filename), flags=cv2.IMREAD_UNCHANGED)

depth_image = du.transform_depth_image(depth_image, DEPTH_IMAGE_SCALE_PX2MM, max_depth_mm=5000.0)

# Compute Sobel derivatives on depth_image:
sobelx = cv2.Sobel(depth_image, cv2.CV_64F, 1, 0, ksize=11)
sobely = cv2.Sobel(depth_image, cv2.CV_64F, 0, 1, ksize=11)
magnitude = np.sqrt(sobelx**2 + sobely**2)


magnitude_filtered = magnitude # cv2.GaussianBlur(magnitude, (11, 11), 0)

# plot the magnitude image, clipped at 500 mm range, log-scaled
# clip x range between 200 and 900 pixels in the plot
plt.figure(figsize=(15, 15))
plt.imshow(np.log1p(np.clip(magnitude_filtered, 0, 500000)), cmap='gray')
plt.colorbar()
plt.title('Sobel Magnitude (clipped at 500mm, log-scaled)')
#plt.xlim(200, 900)
plt.show()

print(f"Max Sobel magnitude: {np.max(magnitude)}")

magnitude_ravelled_values = magnitude.ravel()

# Create histogram of Sobel magnitude values
plt.figure(figsize=(15, 15))

# Plot full range histogram
plt.subplot(2, 2, 1)
plt.hist(magnitude_ravelled_values[magnitude_ravelled_values > 0], bins=100)
plt.yscale('log')
plt.title('Full Range Histogram of Sobel Magnitude Values (discarding zero values)')
plt.xlabel('Magnitude')
plt.ylabel('Frequency (log scale)')

# Plot zoomed histogram for values below 50. Discard zero magnitude values.
plt.subplot(2, 2, 2)

zoom_in_lower_range_magnitudes = magnitude_ravelled_values[(magnitude_ravelled_values > 0) & (magnitude_ravelled_values < 50)]
plt.hist(zoom_in_lower_range_magnitudes, bins=127)
plt.yscale('log')
plt.title('Zoomed Histogram of Sobel Magnitude Values (< 50, Zero Values Discarded)')
plt.xlabel('Magnitude')
plt.ylabel('Frequency')
plt.tight_layout()

deltas = np.diff(np.sort(magnitude_ravelled_values))

# Plot the distribution of deltas
plt.subplot(2, 2, 3)
plt.hist(deltas[deltas > 0], bins=100)
plt.yscale('log')  # Set y-axis to logarithmic scale
plt.title('Distribution of Deltas Between sorted magnitude values (discarding zero deltas)')
plt.xlabel('Delta')
plt.ylabel('Frequency')
plt.yscale('log')

# Zoom in to previous plot
plt.subplot(2, 2, 4)
plt.hist(deltas[(deltas > 0) & (deltas < 15)], bins=100)
plt.yscale('log')  # Set y-axis to logarithmic scale
plt.title('Distribution of Deltas Between sorted magnitude values (zoomed in, discarding zero deltas)')
plt.xlabel('Delta')
plt.ylabel('Frequency')
plt.yscale('log')
plt.show()

def display_depth_image(depth_image):
    # Display image:
    plt.figure(figsize=(15, 15))
    img_plot = plt.imshow(depth_image)
    plt.colorbar(img_plot, label='Depth value')
    plt.title('Depth Image with Color Grade Key')
    #plt.axis('off')
    plt.show()
display_depth_image(depth_image)

In [ ]:
# Just a scratchpad to test the du.compose_grey_plus_hillshade_depth_image function used 
# in prepare_data.py for preparing data prior to YOLO model training

dir = "/mnt/061A31701A315E3D/ipd-dataset/phase2-dataset-yolo11-all-objects/images/train"
# list all png files in this directory:

png_files = glob.glob(os.path.join(dir, "*.png"))
for png_file in png_files[:10]:
    img = cv2.imread(png_file, flags=cv2.IMREAD_UNCHANGED)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    grey_img = img[..., 0]
    hillshade_img_0 = img[..., 1]
    hillshade_img_135 = img[..., 2]

    # plot the three images:
    plt.figure(figsize=(15, 5))
    plt.subplot(1, 3, 1)
    plt.imshow(grey_img, cmap='gray')
    plt.title('Grey Image')
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.imshow(hillshade_img_0, cmap='gray')
    plt.title('Hillshade Image 0')
    plt.axis('off')

    plt.subplot(1, 3, 3)
    plt.imshow(hillshade_img_135, cmap='gray')
    plt.title('Hillshade Image 135')
    plt.axis('off')

    plt.show()

In [ ]:
# This cell analyzes and visualizes the depth compression/decompression functions used to encode depth values into single bytes.
# It plots:
# 1. The compression function: y = floor(41.0 * ln(x+1)) which maps depth values in [0--500] mm to 0-255 range
# 2. The decompression function: x = exp(y/41.0) - 1 which recovers the original depth values
# 3. The identity line y=x for reference
# 4. The relative error between original and decompressed values
# This helps validate the information preservation of the compression scheme used for depth images.

# Plot functions
# given a = 41.0
# y = floor( a * ln(x+1) )  # <-- the depth compression function to encode depth values in millimeters, to a single byte
# x = exp(y / a) - 1        # <-- the depth decompression function to decode depth values from a single byte, to millimeters

# Plot the functions:
a = 41.0
x_coords = np.linspace(0, 500, 1000) # Original x-values for plotting
y_compressed = np.floor(a * np.log(x_coords + 1))
y_decompressed = np.exp(y_compressed / a) - 1

# Create figure and primary axis
fig, ax1 = plt.subplots(figsize=(10, 6)) # Adjust figsize as needed

# Plot functions on the primary y-axis (ax1)
ax1.plot(x_coords, y_compressed, label='Compressed Depth')
ax1.plot(x_coords, y_decompressed, label='Decompressed Depth')
ax1.plot(x_coords, x_coords, label='Identity (y=x)', linestyle=':', color='gray') # Identity line

ax1.set_xlabel("Original Depth (x) [mm]")
ax1.set_ylabel("Depth Values")
ax1.legend(loc='center left') # Legend for ax1 plots
ax1.grid(True)

# Create the secondary y-axis (ax2) for the relative error
ax2 = ax1.twinx()

# Calculate relative error: |x_coords - y_decompressed| / x_coords
# Handle division by zero at x_coords = 0 where error is 0.
# y_decompressed[0] is 0 when x_coords[0] is 0.
# So relative_error[0] will be 0 if denominator is non-zero.
denominator = np.maximum(x_coords, 1e-9) # Avoid division by zero; use small epsilon if x_coords is zero.
relative_error = np.abs(x_coords - y_decompressed) / denominator

# Plot relative error on ax2
ax2.plot(x_coords, relative_error, color='red', linestyle='--', label='Relative Error')
ax2.set_ylabel('Relative Error', color='red') # Label for the right y-axis
ax2.tick_params(axis='y', labelcolor='red') # Color for ticks on right y-axis
ax2.set_ylim(0.0, 0.2) # Set y-axis range for relative error
ax2.legend(loc='upper right') # Legend for ax2 plot

# Add a title to the entire plot
plt.title("Depth Compression, Decompression, and Relative Error Analysis")

fig.tight_layout() # Adjust plot layout to prevent labels/titles from overlapping
plt.show()


In [8]:
# yolo_detectors_single_object:  the old approach: one model per object class.
# yolo_detector_multiclass: the new approach single model for all object classes approach with hillshade depth mapping

yolo_detectors_single_object = {}
object_meshes = {}

for this_obj_id in obj_ids:
    obj_id_path = str(1000000+this_obj_id)[1:]
    ply_file = os.path.join(models_dir, f"obj_{obj_id_path}.ply")
    
    yolo_model_path_phase1 = f'/home/joao/source/bpc-challenge/opencv/bpc/models_phase-1-backup/detection/obj_{this_obj_id}/yolo11-detection-obj_{this_obj_id}.pt'
    yolo_model_path_phase2 = f'/home/joao/source/bpc-challenge/opencv/bpc/models_phase-2-backup/detection/obj_{this_obj_id}/yolo11-detection-obj_{this_obj_id}.pt'
    
    
    if PHASE == 1:
        yolo_model_path = yolo_model_path_phase1
    elif PHASE == 2:
        yolo_model_path = yolo_model_path_phase2

    yolo_detectors_single_object[this_obj_id] = YOLODetector(yolo_model_path, None, this_obj_id)
    object_meshes[this_obj_id] = obj = trimesh.load(ply_file)

yolo_model_phase2_multiclass_dir = "/home/joao/source/bpc-challenge/opencv/bpc/models/detection"
yolo_model_name = "yolo-11-training-08-single-model-grey-plus-depth-hillshade.pt"
yolo_detection_thresholds_name = "detection_confidence_thresholds.json"
object_detector = ObjectDetector(os.path.join(yolo_model_phase2_multiclass_dir, yolo_model_name),
                                 os.path.join(yolo_model_phase2_multiclass_dir, yolo_detection_thresholds_name),
                                 is_synthetic=True) # set to use synthetic data from phase 2 PBR training dataset only - there's no real dataset

In [ ]:
TOTAL_IMAGES_TO_PROCESS = 10
USE_OLD_YOLO_SINGLE_CLASS_APPROACH = False

for image_id in image_ids[:TOTAL_IMAGES_TO_PROCESS]:
    dummy_obj_id = 99999

    # Get only camera 1 (index 0):
    capture = Capture.from_dir(scene_dir, cam_ids, image_id, dummy_obj_id)
    image_cam_1 = capture.images[0].copy()
    #image_cam_1 = du.apply_image_transformations(image_cam_1)
    capture.images[0] = image_cam_1
    image_cam_1 = capture.images[0].copy()
    depth_cam_1_raw_values = capture.depths[0].copy()
    depth_cam_1_metric_mm = du.transform_depth_image(depth_cam_1_raw_values, DEPTH_IMAGE_SCALE_PX2MM, max_depth_mm=5000.0)
    intrinsics_K_cam_1 = capture.Ks[0].copy()

    # Infer for all object IDs at once, then apply inter-class filtering:
    detections_all_obj_ids = {}
  
    if USE_OLD_YOLO_SINGLE_CLASS_APPROACH:
        for this_obj_id in obj_ids:
            detections_this_obj_id = yolo_detectors_single_object[this_obj_id].detect(image_cam_1)
            detections_all_obj_ids[this_obj_id] = detections_this_obj_id
    else: # use the new approach with single YOLO model for all object classes with hillshade depth mapping
        detections_all_obj_ids = object_detector.detect(image_cam_1, depth_cam_1_raw_values)

    excluded_elongated_object_ids=[4, 8, 9]
    detections_all_obj_ids = ydf.filter_detections(detections_all_obj_ids, depth_cam_1_metric_mm, object_meshes, intrinsics_K_cam_1, excluded_elongated_object_ids)

    # Now we can draw the filtered detections on the image:
    for this_obj_id, detections_this_id in detections_all_obj_ids.items():
        # For each detection, draw the object ID, bounding box, confidence:
        for detection_this_id in detections_this_id:
            bbox = detection_this_id['bbox']
            confidence = detection_this_id['confidence']
            bb_center = detection_this_id['bb_center']

            text = f"ID: {this_obj_id} Conf: {confidence:.2f}"
            font = cv2.FONT_HERSHEY_SIMPLEX
            font_scale = 1.4  # Increased by 100%
            thickness = 2
            (text_width, text_height) = cv2.getTextSize(text, font, font_scale, thickness)[0]
            
            # Set the background color for the text
            bg_color = du.get_color_for_class_id(this_obj_id)
            text_color = (255, 255, 255)  # White color for the text

            # Calculate the rectangle coordinates for the background
            rect_x = int(bbox[0])
            rect_y = int(bbox[1] - 50)  # Position the rectangle above the bounding box
            rect_w = text_width + 10
            rect_h = text_height + 20

            # Draw the background rectangle
            cv2.rectangle(image_cam_1, (rect_x, rect_y), (rect_x + rect_w, rect_y + rect_h), bg_color, -1)

            # Draw the object bounding box
            cv2.rectangle(image_cam_1, (int(bbox[0]), int(bbox[1])), (int(bbox[2]), int(bbox[3])), du.get_color_for_class_id(this_obj_id), 4)

            # Put the text on the image
            cv2.putText(image_cam_1, text, (int(bbox[0]) + 5, int(bbox[1] - 10)), font, font_scale, text_color, thickness)

    # Show the image with detections
    plt.figure(figsize=(15, 15))
    plt.imshow(image_cam_1)
    plt.axis('off')
    plt.show()


In [ ]:
img_sample_dir = "/home/joao/source/bpc-challenge/opencv/bpc/ipd/val/000000/"
img_grey_path = os.path.join(img_sample_dir, "rgb_cam1", "000002.png")
depth_path = os.path.join(img_sample_dir, "depth_cam1", "000002.png")
img_grey = cv2.imread(img_grey_path, flags=cv2.IMREAD_UNCHANGED)
img_depth = cv2.imread(depth_path, flags=cv2.IMREAD_UNCHANGED)

# Plot both images:
plt.figure(figsize=(15, 15))
plt.subplot(121)
plt.imshow(img_grey)
plt.title('Grayscale Image')
plt.axis('off')

plt.subplot(122) 
plt.imshow(img_depth)
plt.title('Depth Image')
plt.axis('off')
plt.show()

x_range = (860, 3180)

yolo_input = du.compose_grey_plus_hillshade_depth_image(img_grey, img_depth, x_range, is_synthetic=False)

# Plot each channel of yolo input separately, in a 3x1 subplot:
plt.figure(figsize=(15, 5))

plt.subplot(131)
plt.imshow(yolo_input[:,:,0], cmap='gray')
plt.title('Channel 0 - Grayscale')
plt.axis('off')

plt.subplot(132)
plt.imshow(yolo_input[:,:,1], cmap='gray') 
plt.title('Channel 1 - Hillshade 0°')
plt.axis('off')

plt.subplot(133)
plt.imshow(yolo_input[:,:,2], cmap='gray')
plt.title('Channel 2 - Hillshade 135°') 
plt.axis('off')

plt.show()


In [11]:
if False:
    # scene_dir = "./datasets/ipd_bop_data_jan25_1/train_pbr/000000/"
    # models_dir = './datasets/ipd_bop_data_jan25_1/models_eval/'
    scene_dir = "./datasets/ipd/test/000003/"
    models_dir = './datasets/ipd/models_eval/'
    cam_ids = ["cam1", "cam2", "cam3"]
    image_id = 2
    this_obj_id = 11
    obj_id_path = str(1000000+this_obj_id)[1:]
    ply_file = os.path.join(models_dir, f"obj_{obj_id_path}.ply")
    obj = trimesh.load(ply_file)
    yolo_model_path = f'bpc/yolo/models/detection/obj_{this_obj_id}/yolo11-detection-obj_{this_obj_id}.pt'
    pose_model_path = f'bpc/pose/pose_checkpoints/obj_{this_obj_id}/final_model.pth'

    pose_params = PoseEstimatorParams(
        yolo_model_path=yolo_model_path,
        pose_model_path=pose_model_path, 
        yolo_conf_thresh=0.01,
    )
    pose_estimator = PoseEstimator(pose_params)
    t = time.time()
    capture = Capture.from_dir(scene_dir, cam_ids, image_id, this_obj_id)
    detections_this_id = pose_estimator._detect(capture)
    pose_predictions = pose_estimator._match(capture, detections_this_id)
    pose_estimator._estimate_rotation(pose_predictions)
    print(time.time() - t)

    for idx in range(len(capture.Ks)):
        plt.figure(figsize=(15, 15))
        plt.imshow(capture.images[idx])
        a, b = render_mask(obj, capture.Ks[idx], (capture.RTs[idx]), capture.images[0].shape[:2][::-1], [x.pose for x in pose_predictions])
        plt.imshow(a, alpha=0.5)
        plt.show()
